# 01 — Stockholm summer heat-event screening

This notebook runs the matching Python script, downloads **real SMHI hourly air temperature** for 2019–2023, and displays detected events. Requires network access from CSC. The first analysis uses a **user-selected screening definition** (3 consecutive days with daily maxima ≥25°C), **not** official SMHI heat warnings or corridor-scale heat stress.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Run Jupyter from the project root, or this cell locates the script relative to notebooks/.
root = Path.cwd()
if not (root / "scripts" / "01_stockholm_heatwaves.py").exists():
    root = root.parent
assert (root / "scripts" / "01_stockholm_heatwaves.py").exists(), "Open notebook from repo root or notebooks/"
sys.path.insert(0, str(root / "scripts"))
from importlib import import_module
heat = import_module("01_stockholm_heatwaves")
print("Project root:", root)

Project root: /scratch/project_2010417/paper4/stockholm-github


In [2]:
YEARS = [2019, 2020, 2021, 2022, 2023]
THRESHOLD_C = 25.0
MIN_CONSECUTIVE_DAYS = 3
OUT = root / "data" / "processed" / "stockholm_heat"

hourly, daily, events = heat.run(
    years=YEARS, output_dir=OUT,
    threshold=THRESHOLD_C, min_days=MIN_CONSECUTIVE_DAYS
)
print(f"Hourly observations: {len(hourly):,}; events: {len(events)}")
display(events)

/scratch/project_2010417/paper4/stockholm-github/scripts/01_stockholm_heatwaves.py:120: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([archive,recent],ignore_index=True)
/scratch/project_2010417/paper4/stockholm-github/scripts/01_stockholm_heatwaves.py:168: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  daily['hot_day'] = daily.hot_day.fillna(False).astype(bool)


Hourly observations: 11,040; events: 20


,event_id,start_date,end_date,duration_days,peak_tmax_c,mean_tmax_c
0,1,2019-06-05,2019-06-08,4,28.5,27.225000
1,2,2019-07-18,2019-07-20,3,27.0,26.000000
2,3,2019-07-24,2019-07-29,6,31.3,28.516667
3,4,2019-08-26,2019-08-28,3,27.4,27.066667
4,5,2020-06-23,2020-06-28,6,30.9,29.633333
5,6,2020-07-16,2020-07-19,4,26.3,26.150000
6,7,2020-08-07,2020-08-09,3,29.9,28.500000
7,8,2020-08-12,2020-08-22,11,29.9,27.227273
8,9,2021-06-05,2021-06-07,3,27.1,25.800000
9,10,2021-06-18,2021-06-22,5,29.3,28.940000


In [3]:
display(daily.groupby("year").agg(observed_days=("eligible", "sum"), hot_days=("hot_day", "sum"), peak_temp_c=("tmax_c", "max")))
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily["local_date"], daily["tmax_c"], lw=0.8)
ax.axhline(THRESHOLD_C, linestyle="--", label="Screening threshold")
for _, ev in events.iterrows():
    ax.axvspan(pd.Timestamp(ev.start_date),pd.Timestamp(ev.end_date)+pd.Timedelta(days=1),alpha=0.15)
ax.set(xlabel="Date (Stockholm local)", ylabel="Daily max air temperature (°C)", title="Summer heat episodes — SMHI station")
ax.legend(); plt.show()

,observed_days,hot_days,peak_temp_c
year,,,
2019,92,24,31.3
2020,92,28,30.9
2021,92,32,32.9
2022,92,30,34.0
2023,92,16,29.3


## Next steps
- Verify station selection and missing-data coverage in `data/processed/stockholm_heat/analysis_metadata.json`.
- Repeat with alternative thresholds and sensitivity tests.
- Align event hours with Marina’s available summer mobility periods and GTFS snapshots.
- Do **not** interpret point-station air temperature as street-level UTCI or individual exposure.